In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from toponymy.annotation import NodeId, AnnotationTree, Annotation, AnnotationStore, Executor, AnnotationState

15:03:03 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
15:03:03 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


In [3]:
import numpy as np

In [4]:
from toponymy.tools.notebook_data_load import load_small_newsgroups
newsgroups_df = load_small_newsgroups()
embeddings = np.stack(newsgroups_df["embedding"].values)
document_map = np.stack(newsgroups_df["map"].values)

In [5]:
from toponymy import ToponymyClusterer
clusterer = ToponymyClusterer(min_clusters=4, verbose=True)
clusterer.fit(document_map, embeddings);

Layer 0 found 8 clusters
Layer 1 found 4 clusters


In [6]:
clusterer.cluster_tree_

{(1, 0): [(0, 0)],
 (1, 1): [(0, 1)],
 (1, 2): [(0, 7), (0, 6), (0, 5)],
 (1, 3): [(0, 4)],
 (2, 0): [(1, 0), (1, 1), (1, 2), (1, 3), (0, 2), (0, 3)]}

In [7]:
objects = newsgroups_df["post"].str.strip().values
embedding_vectors = embeddings
clusterable_vectors = document_map

## AnnotationStore

In [8]:
store = AnnotationStore.from_clusterer(clusterer, objects, embedding_vectors)

In [9]:
executor = Executor(store)

## Compute exemplars

In [10]:
store

AnnotationStore(cluster_object_indices[12/12], cluster_objects[12/12], cluster_centroid[12/12], cluster_object_vectors[12/12])

In [11]:
layer = clusterer.cluster_layers_[0]

In [12]:
null_topic = np.mean(embedding_vectors, axis=0)

In [13]:
from toponymy.exemplar_texts import DiverseExemplarAnnotator

In [14]:
failures = executor.run(DiverseExemplarAnnotator(null_topic=null_topic))

In [15]:
executor.store

AnnotationStore(cluster_object_indices[12/12], cluster_objects[12/12], cluster_centroid[12/12], cluster_object_vectors[12/12], exemplars[12/12], exemplar_original_indices[12/12])

In [16]:
cluster = store.node(NodeId(0, 0))
[f"{i}: {text[:30]}..." for i, text in zip(cluster["exemplar_original_indices"], cluster["exemplars"])]

['74: Well, it looks like, just as D...',
 '66: OK, you guys stirred up my chi...',
 '101: #21\tPETER AHOLA\t\tSeason: 2nd\nA...',
 '54: Is there any judgement call on...']

## Topic Annotator

WARNING: This requires a running LLM

In [17]:
from toponymy.topic_naming import TopicNamerTemplate

In [18]:
from toponymy.templates import MULTILINGUAL_EN_FR_PROMPT_TEMPLATES

In [19]:
template = TopicNamerTemplate(object_description="newsgroup posts",
                              corpus_description="20-newsgroups dataset",
                              tree=store.tree,
                              exemplar_start_delimiter="<EXAMPLE_POST>\n",
                              exemplar_end_delimiter="\n</EXAMPLE_POST>\n\n",
                              prompt_template=MULTILINGUAL_EN_FR_PROMPT_TEMPLATES)

In [20]:
template.parse('{"english_topic_name": "Baseball Statistics and Analysis", "nom_du_sujet_en_français": "French", "topic_specificity": 0.9}')

TopicParseResult(topic_payload='Baseball Statistics and Analysis / French', raw_response='{"english_topic_name": "Baseball Statistics and Analysis", "nom_du_sujet_en_français": "French", "topic_specificity": 0.9}', error=None)

In [21]:
from toponymy.topic_naming import TopicNameAnnotator

In [22]:
from toponymy.llm_wrappers import OllamaNamer, AsyncOllamaNamer
from toponymy.debug_logging import BasicDebugLogger
from toponymy.tools.notebook_data_load import notebook_output_dir

debug_logger_file = notebook_output_dir() / "llm_debug_topic_annotation.jsonl"
debug_logger_file.write_text("", encoding="utf-8")  # Clear the file before starting
debug_logger = BasicDebugLogger(debug_logger_file, truncate=True)

namer = OllamaNamer(callback=debug_logger)
namer.test_llm_connectivity()  # Verify connection
namer.connectivity_status()

{'success': True,
 'model': 'ollama_chat/llama3.2',
 'wrapper': 'LiteLLMNamer',
 'response': '{"status": "ok"}',
 'error_type': None,
 'error_message': None,
 'original_exception': None}

In [23]:
executor.store

AnnotationStore(cluster_object_indices[12/12], cluster_objects[12/12], cluster_centroid[12/12], cluster_object_vectors[12/12], exemplars[12/12], exemplar_original_indices[12/12])

In [24]:
%%time
failures_sync = executor.run(TopicNameAnnotator(llm_namer=namer, topic_name_template=template))

CPU times: user 101 ms, sys: 20.2 ms, total: 121 ms
Wall time: 22.6 s


In [25]:
failures_sync

{NodeId(0, 0): 'LLMOutputParseError: No match for regex in LLM output (118 chars):\n{"Baseball Statistics and Analysis": "Analyse des statistiques et de l\'analyse du baseball", "topic_specificity": 0.9}\n',
 NodeId(0, 2): 'LLMOutputParseError: No match for regex in LLM output (117 chars):\n{"cryptography": "Cryptography and Security", "cryptographie": "Cryptographie et Sécurité", "topic_specificity": 0.9}\n',
 NodeId(1, 0): 'LLMOutputParseError: No match for regex in LLM output (78 chars):\n{"baseball statistics": "statistiques de base-ball", "topic_specificity": 0.9}\n'}

In [26]:
import time

In [27]:
start = time.perf_counter()
failures = await executor.arun(TopicNameAnnotator(llm_namer=AsyncOllamaNamer(), topic_name_template=template))
print(f"Elapsed: {time.perf_counter() - start:.4f}s")

Elapsed: 24.8893s


In [28]:
failures

{NodeId(0, 7): 'LLMOutputParseError: No match for regex in LLM output (164 chars):\n{"english_topic_name": "Christian Theology and Ethics", "nom_du_sujet_en_fran\\u00e7ais": "Th\\u00e9ologie chr\\u00e9tienne et \\u00e9thique", "topic_specificity": 0.9}\n'}

In [29]:
store.draft_topics.states

mappingproxy({NodeId(0, 0): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 1): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 2): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 3): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 4): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 5): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 6): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 7): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(1, 0): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(1, 1): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(1, 2): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(1, 3): <AnnotationState.COMPUTED: 'computed'>})

In [30]:
for node in store.tree.nodes:
    if store.draft_topics.states[node] == AnnotationState.COMPUTED:
        print(node, store.draft_topics[node])
    else:
        print(node, store.draft_topics.states[node])

(0, 0) Baseball Statistics and Analysis / Statistiques et analyse du baseball
(0, 1) Computer Hardware and Technology / Informatique et matériel informatique
(0, 2) Cryptography / Cryptographie
(0, 3) Fungal Infections and Immunology / Infections fongiques et immunologie
(0, 4) Car Maintenance and Security / Entretien et sécurité de la voiture
(0, 5) Criticism of Government Policies and Corporate Practices / Critique des politiques gouvernementales et des pratiques des entreprises
(0, 6) Anti-Zionism and Anti-Semitism / Antisionisme et antisémitisme
(0, 7) Theological Debates on Original Sin and Divine Right / Débats théologiques sur le péché original et le droit divin
(1, 0) Baseball Statistics and Analysis / Statistiques et analyse du baseball
(1, 1) Computer Hardware and Technology / Informatique et matériel informatique
(1, 2) Anti Zionism / Antisionisme
(1, 3) Car Maintenance and Security / Entretien et sécurité de la voiture


In [31]:
from pathlib import Path
import json

In [32]:
def load_multiline_json_logs(path: str | Path) -> list[dict]:
    """
    Helper function to load a JSONL file.

    Args:
        path (str | Path): Path to the JSONL file.

    Returns:
        list[dict]: List of JSON objects loaded from the file.
    """
    path = Path(path)
    records = []

    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Skipping bad JSON on line {i}: {e}")
    return records



In [33]:
debug_logs = load_multiline_json_logs(debug_logger_file)

In [34]:
for log in debug_logs:
    print(log["raw_response"])

{"Baseball Statistics and Analysis": "Analyse des statistiques et de l'analyse du baseball", "topic_specificity": 0.9}
{"english_topic_name": "Computer Hardware and Software Discussion", "nom_du_sujet_en_français": "Discussion sur les logiciels et les composants informatiques", "topic_specificity": 0.8}
{"cryptography": "Cryptography and Security", "cryptographie": "Cryptographie et Sécurité", "topic_specificity": 0.9}
{"english_topic_name": "Medical Myths and Misconceptions", "nom_du_sujet_en_français": "Mythes et malentendus médicaux", "topic_specificity": 0.8}
{"english_topic_name": "Car Maintenance and Convertible Safety", "nom_du_sujet_en_français": "Maintenance et sécurité des voitures convertibles", "topic_specificity": 0.85}
{"english_topic_name": "Anti Government Criticism and Economic Policy", "nom_du_sujet_en_français": "Critique du gouvernement et de la politique économique", "topic_specificity": 0.85}
{"english_topic_name": "Middle East Politics and Conflict", "nom_du_suje